# Aquaplanet

This notebook demonstrates a JAX-ESM (JEM) example using JAX-GCM (JCM) and Slab Ocean Model as an aquaplanet.

In [ ]:
from pathlib import Path

import jcm
from jcm.physics.speedy.speedy_coords import get_speedy_coords
import jax_datetime as jdt

from jem.base.coupler import Coupler
from jem.components import JCMComponent, SlabOceanModel, SlabSeaiceModel
from jem.components.slab import SlabGrid

use_ipython = 'get_ipython' in globals()

## Configurations

In [ ]:
start_datetime = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")
simulation_name = "01-01_aquaplanet"
output_dir = (Path("output") / simulation_name).resolve()
output_dir.mkdir(exist_ok=True, parents=True)
output_figure = output_dir / "animation_humidity_sst.gif"

## Creating Flux and Scalar Exchange between Components

An *exchanger* is the only place where components exchange information. It
receives the mapping of every component's carry together with the coupler's
clock, and returns the mapping to continue with. It is traced with the rest of
the coupled step, so it must not write into the carries it is handed: it builds
new ones with `.replace(...)` and returns them.

In [ ]:
def exchange(components, time):
    del time  # this exchange does not depend on the date

    atm = components["atm"]
    ocn = components["ocn"]
    seaice = components["seaice"]

    ocn = dict(ocn, forcing=ocn["forcing"].replace(
        total_heat_flux=atm["derived"].total_heat_flux,
    ))
    seaice = dict(seaice, forcing=seaice["forcing"].replace(
        ice_frazil_melt_energy=ocn["derived"].ice_frazil_melt_energy,
    ))
    atm = dict(atm, forcing=atm["forcing"].replace(
        sea_surface_temperature=ocn["state"].sea_surface_temperature,
        sice_am=seaice["derived"].ice_fraction,
    ))

    return dict(components, atm=atm, ocn=ocn, seaice=seaice)

## Create Components

In [ ]:
atm_model = jcm.model.Model(
    coords=get_speedy_coords(),  # T31 spectral resolution with 8 vertical levels
    start_date=start_datetime,
)

# The slab grid is built from the atmosphere's own horizontal grid, so the two
# cannot end up on grids that merely resemble each other. Aquaplanet: no
# fractional mask, so every cell is ocean.
aquaplanet_grid = SlabGrid.from_coords(atm_model.coords.horizontal)

# The coupler owns the clock: the coupling timestep, start date and calendar
# live here, and every component's step is handed the same CouplingTime. It
# calls JCMComponent.bind(), which checks that the coupling timestep is a whole
# multiple of JCM's own and that the two agree on the start date and calendar.
model = Coupler(
    dict(
        atm=JCMComponent(atm_model),
        ocn=SlabOceanModel(aquaplanet_grid),
        seaice=SlabSeaiceModel(aquaplanet_grid, name="seaice"),
    ),
    dict(exchange=exchange),
    coupling_timestep=coupling_timestep,
    start_date=start_datetime,
)

# The workflow defaults to every exchanger followed by every component: the
# fields are exchanged first, then all three components step on the same state.
print(repr(model))

## Run Coupled Model

In [ ]:
simulation_interval = jdt.to_timedelta(90, "day")
run = model.generate_trajectory_function(
    int(simulation_interval / coupling_timestep)
)
initial_carry = model.initialize()
final_carry, diagnostics = run(initial_carry)

## Output into NetCDF

In [ ]:
output_dict = model.to_xarray(diagnostics)
output_dict_subsample = {}
subsample_skip = 5
for component_name, ds in output_dict.items():
    output_file = output_dir / f"{component_name:s}.nc"
    print(f"Output file: {str(output_file)}, with subsample_skip = {subsample_skip:d}")
    ds = ds.isel(time=slice(None, None, subsample_skip))
    ds.to_netcdf(output_file, engine="netcdf4")
    output_dict_subsample[component_name] = ds

## Visualization: animation of specific humidity of the surface grid

In [ ]:

import matplotlib as mplt
if not use_ipython:
    mplt.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
import numpy as np

output_dict_animation = {
    component_name: _ds.isel(time=slice(None, None, 1))
    for component_name, _ds in output_dict_subsample.items()
}

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.gridlines(draw_labels=True)
cb = None
cf = None
cs = None
ch = None

def update(frame):
    print(f"Plotting frame={frame:d}")
    global cf, cb, cs, ch
    _data_q = output_dict_animation["atm"]["specific_humidity"].isel(time=frame, level=0)
    _data_sst = output_dict_animation["ocn"]["sea_surface_temperature"].isel(time=frame) - 273.15
    _data_sit = output_dict_animation["seaice"]["ice_thickness"].isel(time=frame)
    coords = _data_q.coords
    time_str = _data_q['time'].dt.strftime('%Y-%m-%d').to_numpy().item()
    lat = coords["lat"]
    lon = coords["lon"]

    # Remove previous frame's artists before drawing the new ones
    cf and cf.remove()
    cs and cs.remove()
    ch and ch.remove()
    
    # Plot the humidity field for the current time step
    cyclic_data_q, cyclic_lon = add_cyclic_point(_data_q.to_numpy().transpose(), coord=lon)
    mappable = ax.contourf(
        cyclic_lon, lat,
        cyclic_data_q,
        levels=1 + np.linspace(0, 1, 21) * 10,
        transform=ccrs.PlateCarree(), 
        cmap='GnBu',
        extend="both",
    )
    
    cyclic_data_sst, cyclic_lon = add_cyclic_point(_data_sst.to_numpy().transpose(), coord=lon)
    cs = ax.contour(
        cyclic_lon, lat,
        cyclic_data_sst,
        levels=np.arange(-2, 31, 4),
        transform=ccrs.PlateCarree(),
        colors="black",
    )
    ax.clabel(cs, fontsize=12)

    # Dot-hatch grid cells that carry any sea ice (thickness above zero)
    cyclic_data_sit, cyclic_lon = add_cyclic_point(_data_sit.to_numpy().transpose(), coord=lon)
    ch = ax.contourf(
        cyclic_lon, lat,
        cyclic_data_sit,
        levels=[1e-6, np.inf],
        colors="none",
        hatches=["."],
        transform=ccrs.PlateCarree(),
    )

    ax.set_title(f"[{time_str:s}]\nSurface specific humidity (shading) and sea surface temperature (contours, ${{}}^\\circ \\mathrm{{C}}$),\nwith sea ice (dotted hatching)")
    if cb is None:
        cb = plt.colorbar(ax=ax, mappable=mappable, orientation='vertical', shrink=0.7, pad=0.07)
        cb.set_label("[g/kg]", fontsize=12)
    
    return [cf,]
    
# Generate and save
ani = FuncAnimation(fig, update, frames=len(output_dict_animation["atm"].coords["time"]), interval=120, blit=False)
print("Saving animation: ", output_figure)
ani.save(output_figure, writer='pillow', dpi=200)

if use_ipython:
    from IPython.display import Image
    display(Image(output_figure))